# Phase 4 -- Handling Class Imbalance with SMOTE

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('credit_card_fraud_dataset.csv')
print(f"Initial dataset shape: {df.shape}")

Initial dataset shape: (500, 17)


### 4.1 -- Re-apply Encoding & Feature Splitting

We drop transaction_id and encode categoricals exactly as done in Phase 3.

In [2]:
# Encode day of week
day_mapping = {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}
df['day_of_week'] = df['day_of_week'].map(day_mapping)

# Encode merchant category
le_merchant = LabelEncoder()
df['merchant_category'] = le_merchant.fit_transform(df['merchant_category'])

# Separate features and target
df.drop('transaction_id', axis=1, inplace=True)
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

print("Features X shape:", X.shape)
print("Target y shape:", y.shape)

Features X shape: (500, 15)
Target y shape: (500,)


### 4.2 -- Train/Test Split

We split the data before scaling and SMOTE to avoid data leakage.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (400, 15)
X_test:  (100, 15)


### 4.3 -- Standardize Numerical Features

Scale train and test using scaler fit only on training set.

In [4]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

### 4.4 -- Apply SMOTE

Apply SMOTE only on training data to address class imbalance.

In [5]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Oversampling completed.")

Oversampling completed.


### 4.5 -- Verify Class Distribution and Shapes

In [6]:
print("=== Before SMOTE ===")
print(f"X_train shape: {X_train_scaled.shape}")
print(f"y_train distribution:\n{y_train.value_counts()}")

print("\n=== After SMOTE ===")
print(f"X_train_resampled shape: {X_train_resampled.shape}")
print(f"y_train_resampled distribution:\n{y_train_resampled.value_counts()}")

=== Before SMOTE ===
X_train shape: (400, 15)
y_train distribution:
is_fraud
0    380
1     20
Name: count, dtype: int64

=== After SMOTE ===
X_train_resampled shape: (760, 15)
y_train_resampled distribution:
is_fraud
0    380
1    380
Name: count, dtype: int64


---
### Phase 4 Summary

- Class imbalance has been addressed by applying SMOTE strictly on the training partition.
- The target distribution is now balanced at 50/50 (380 samples each for genuine and fraud cases in the training set).
- The test partition remains in its original imbalanced format to ensure unbiased evaluation metrics.

Ready for Phase 5 (Model Training & Evaluation).

---